<a href="https://colab.research.google.com/github/sgisgeodata/sgis-data-manual/blob/main/%EA%B0%9C%EB%B3%84%20%EA%B5%90%EC%9C%A1%EC%9E%90%EB%A3%8C%20Training%20Materials/(260716)%20%EA%B4%91%EC%A3%BC%EC%97%B0%EA%B5%AC%EC%9B%90/GeoAI%20%EC%8B%A4%EC%8A%B5/GeoAI_water/geoai_gwangju.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GeoAI를 활용한 Sentinel-2 수체 탐지 실습

이번 실습에서는 Google Earth Engine에서 Sentinel-2 영상을 불러온 뒤, GeoAI 사전학습 모델을 활용하여 수체 영역을 탐지합니다.

### 전체 실습 흐름

`관심 지역 설정 → Sentinel-2 검색 → 구름 제거 → 4밴드 구성 → GeoTIFF 저장 → GeoAI 추론 → 벡터 변환 → 시각화`

Earth Engine 데이터는 Google 계정과 프로젝트 권한을 확인한 뒤 사용할 수 있습니다. 따라서 실습 전 다음 준비가 필요합니다.

- Google 계정 로그인
- [Google Earth Engine ](https://console.cloud.google.com/earth-engine/welcome) 접속
- 구성탭 →  새 프로젝트 만들기 → 프로젝트 ID 설정 → 비상업용으로 등록






## 1. 실습 환경 설치

Google Earth Engine과 수체 탐지에 필요한 패키지를 설치합니다.


In [ ]:
# 실습 패키지 설치
!pip install -q geemap earthengine-api geoai-py overturemaps   # Google Earth Engine
!pip install -q -U geoai-py localtileserver                    # GeoAI

## 2. Google Earth Engine 인증
Google Earth Engine을 사용하기 위해 인증과 초기화를 수행합니다.  

`project` 값은 본인의 Earth Engine 프로젝트 ID에 맞게 수정합니다.
Google 계정 인증 창이 열리면 허용 버튼을 누릅니다.

In [ ]:
# 라이브러리 불러오기
import ee
import geemap

# Earth Engine 인증 및 초기화
ee.Authenticate(auth_mode="colab")
ee.Initialize(project="jm0629")

## 3. 실습 지역 ROI 설정

ROI(Region of Interest)는 분석하려는 **관심 영역**입니다.  
이번 실습에서는 영월 청령포 지역을 사각형으로 지정합니다.

In [ ]:
# 실습 지역 ROI 설정
# 좌표 순서: [서쪽 경도, 남쪽 위도, 동쪽 경도, 북쪽 위도]
roi = ee.Geometry.Rectangle([128.435, 37.166, 128.451, 37.180])

## 4. Sentinel-2 영상 생성

> ### 4.0. Sentinel-2 영상을 사용하는 이유
>
> Sentinel-2는 유럽우주국(ESA)의 다중분광 지구관측 위성입니다.
> 사람의 눈으로 보는 RGB 색상뿐 아니라 근적외선(NIR), 단파적외선(SWIR) 등 여러 파장대의 정보를 제공합니다.
>
> 수체 탐지 모델인 `water_detection.pth`는 RGB와 NIR로 구성된 **4채널 영상**을 입력으로 사용하므로, 해당 밴드를 제공하는 Sentinel-2 위성영상을 활용합니다.
>
> | 입력 순서 | Sentinel-2 밴드 | 파장 영역 | 수체 탐지에서의 역할                                 |
> | ----: | ------------- | ----- | ------------------------------------------- |
> |     1 | B4            | Red   | 토지·식생·수체의 가시광선 반사 특성 제공                     |
> |     2 | B3            | Green | 물의 색과 탁도, 주변 토지의 색상 차이 표현                   |
> |     3 | B2            | Blue  | 얕은 물, 대기 영향, 가시광선 색상 정보 제공                  |
> |     4 | B8            | NIR   | 물은 NIR을 강하게 흡수하고 식생은 강하게 반사하므로 수체 구분에 매우 중요 |
>
> RGB만 사용하면 그림자, 어두운 지표면, 건물 지붕 등이 물처럼 보일 수 있지만
> NIR을 추가하면 물과 식생·토지의 분광 차이가 커져 수체를 더 안정적으로 구분할 수 있습니다.


###4.1. 구름 제거 마스킹 함수

구름은 지표면을 가리기 때문에 수체 탐지 결과에 큰 오류를 만들 수 있습니다.  
Sentinel-2의 `QA60` 품질 밴드에는 구름과 권운 여부가 비트값으로 기록되어 있습니다.

- 비트 10: 불투명 구름(Opaque cloud)
- 비트 11: 권운(Cirrus cloud) ** 5~13km의 아주 높은 고도(상층운)에 형성되는 구름

두 비트가 모두 0인 픽셀만 남겨 구름을 마스킹합니다.

In [ ]:
# Sentinel-2 구름 마스킹 함수
def mask_s2_clouds(image):
    qa = image.select("QA60") # QA60 품질관리 밴드 선택

    cloud_bit_mask = 1 << 10  # 비트 10: 불투명 구름
    cirrus_bit_mask = 1 << 11 # 비트 11: 권운

    # 두 비트가 모두 0인 픽셀(=구름이 없는 픽셀)만 마스크로 저장

    mask = (
        qa.bitwiseAnd(cloud_bit_mask).eq(0)
        .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    )

    return image.updateMask(mask)

### 4.2. Sentinel-2 영상 컬렉션 검색

Earth Engine의 `COPERNICUS/S2_SR_HARMONIZED` 컬렉션을 사용합니다.

검색 조건은 다음과 같습니다.

1. ROI와 겹치는 영상
2. 2024년 4월 1일부터 10월 31일까지 촬영된 영상
3. 장면 전체 구름 비율이 20% 미만인 영상
4. QA60 기반 구름 마스킹 적용

In [ ]:
# Sentinel-2 영상 컬렉션 생성
s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(roi)
    .filterDate("2024-04-01", "2024-10-31")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
    .map(mask_s2_clouds)
)

### 4.3. 구름 비율이 가장 낮은 영상 선택

s2 영상 컬렉션 중에 구름 비율이 가장 낮은 영상을 선택하는 과정입니다.


In [ ]:
# 구름 비율이 가장 낮은 영상 선택
best_image = ee.Image(s2.sort("CLOUDY_PIXEL_PERCENTAGE").first())

# 선택 영상 정보 확인
print(
    "선택된 영상 날짜:",
    ee.Date(best_image.get("system:time_start")).format("YYYY-MM-dd").getInfo()
)

print(
    "구름 비율:",
    best_image.get("CLOUDY_PIXEL_PERCENTAGE").getInfo()
)

### 4.4. 모델 입력용으로 전처리

선택한 Sentinel-2 영상에서 다음 순서로 밴드를 추출합니다.

`B4(Red) → B3(Green) → B2(Blue) → B8(NIR)`

이 순서는 단순한 표시 순서가 아니라 **모델 입력 채널 순서**입니다.


In [ ]:
# RGB+NIR 4밴드 구성
s2_rgbnir = (
    best_image
    .select(["B4", "B3", "B2", "B8"])  # Red, Green, Blue, NIR
    .clip(roi)
)

Sentinel-2 반사도 값을 0~255 범위의 8비트 영상으로 변환합니다.

In [ ]:
# GeoAI 모델 입력용 uint8 변환
s2_rgbnir_uint8 = (
    s2_rgbnir
    .clamp(0, 3000)
    .divide(3000)
    .multiply(255)
    .toUint8()
)

## 5. Sentinel-2 영상 시각화

`geemap`을 이용하여 생성한 Sentinel-2 영상을 지도 위에 RGB로 시각화합니다.



In [ ]:
# Sentinel-2 RGB 영상 시각화
m = geemap.Map(center=[37.173, 128.443], zoom=15)

m.addLayer(
    s2_rgbnir_uint8,
    {
        "bands": ["B4", "B3", "B2"],
        "min": 0,
        "max": 255
    },
    "Sentinel-2 RGB"
)

m

## 6. Sentinel-2 영상 GeoTIFF 저장

GeoAI 모델에 입력하기 위해 Sentinel-2 영상을 GeoTIFF 파일로 저장합니다.


In [ ]:
# Sentinel-2 GeoTIFF 저장 경로
raster_path = "/content/cheongnyeongpo_sentinel2_rgbnir.tif"

# Sentinel-2 영상을 GeoTIFF로 저장
geemap.ee_export_image(
    s2_rgbnir_uint8,          # 저장할 Sentinel-2 영상
    filename=raster_path,     # 저장 파일 경로
    scale=10,                 # 공간해상도 10m
    region=roi,               # 저장할 관심 영역
    crs="EPSG:3857",          # 저장 좌표계
    file_per_band=False       # 모든 밴드를 하나의 파일로 저장
)

## 8. GeoAI 수체 탐지 실행

GeoAI의 사전학습 수체 탐지 모델을 적용하여 수체 후보 영역을 추출해보는 실습입니다.


### 8.0. 라이브러리 임포트

In [ ]:
# geoai 라이브러리
import geoai

# 래스터 데이터 처리
import rasterio

# 시각화 라이브러리
import matplotlib.pyplot as plt

# 파일 다운로드를 위한 라이브러리
from google.colab import files

### 8.1. 실습용 위성영상 준비

1) 앞 단계에서 직접 생성한 영상 사용

Earth Engine에서 직접 내보낸 `raster_path`를 사용합니다.  


2) 미리 준비된 실습용 영상 사용





인증 또는 다운로드 문제로 앞 단계를 실행하기 어려운 경우, GitHub 경로에 저장된 GeoTIFF를 내려받아 사용할 수 있습니다.

In [ ]:
# 실습용 위성영상 GitHub URL
raster_url = "https://raw.githubusercontent.com/sgisgeodata/sgis-data-manual/main/개별 교육자료 Training Materials/(260716) 광주연구원/GeoAI 실습/GeoAI_water/cheongnyeongpo_sentinel2_rgbnir.tif"

# 위성영상 다운로드
# raster_path = geoai.download_file(raster_url)

# 위성영상 시각화해서 확인
# geoai.view_raster(raster_url)

### 8.2. 수체탐지 모델 실행

이번 실습에서는 GeoAI의 `object_detection()` 함수를 이용하여 Sentinel-2 영상에서 수체 영역을 탐지합니다.

`water_detection.pth`는 수체와 비수체의 특징을 학습한 사전학습 모델 파일로, RGB와 NIR로 구성된 4밴드 영상을 입력받아 수체로 판단되는 영역을 결과 영상으로 생성합니다.

In [ ]:
# 수체 탐지 결과 저장 경로
prediction_path = "water_prediction.tif"

# GeoAI 수체 탐지 모델 실행
geoai.object_detection(
    raster_path,                         # 입력: 4밴드 Sentinel-2 GeoTIFF
    prediction_path,                     # 수체 탐지 결과 GeoTIFF
    model_path="water_detection.pth",    # 수체 탐지 모델
    window_size=128,                     # 영상을 128×128 픽셀 조각으로 나누어 추론
    overlap=32,                          # 중첩 크기
    confidence_threshold=0.9,            # 탐지 신뢰도 기준
    batch_size=1,                        # 배치 크기(한 번에 처리할 영상 조각 수)
    num_channels=4,                      # 입력 영상 밴드 수: Red, Green, Blue, NIR
)

### 8.3. 수체 탐지 결과 시각화
수체 탐지 결과를 시각화하여 확인합니다.

모델 출력 결과는 픽셀 기반 래스터 마스크입니다.
파란색으로 표시된 부분은 픽셀값이 1인 영역으로, 모델이 수체로 판단한 픽셀입니다.

In [ ]:
# 수체 탐지 결과 불러오기
with rasterio.open(prediction_path) as src:
    prediction = src.read(1)  # 첫 번째 밴드 읽기

# 수체 탐지 결과 시각화
plt.figure(figsize=(8, 8))
plt.imshow(prediction, cmap="Blues")
plt.title("Water Detection Result")
plt.axis("off")
plt.show()

### 8.4. 수체 탐지 결과 벡터 변환 (TIFF → GeoJSON)

픽셀 기반의 수체 탐지 결과를 GeoJSON 폴리곤으로 변환하는 과정입니다.


In [ ]:
# 수체 탐지 결과를 저장할 GeoJSON 경로
geojson_path = "water_prediction.geojson"

# 수체 탐지 래스터를 벡터 폴리곤으로 변환하여 water_gdf에 저장
water_gdf = geoai.raster_to_vector(
    prediction_path,          # 입력: 수체 탐지 결과 TIFF
    geojson_path,             # 출력: GeoJSON 파일
    min_area=1,               # 최소 폴리곤 면적
    simplify_tolerance=1,     # 폴리곤 경계 단순화 정도
)

In [ ]:
# water_gdf 확인
water_gdf.head()

## 10. 수체 탐지 결과를 지도 위에 시각화

원본 Sentinel-2 영상 위에 수체 탐지 결과를 중첩하여 확인합니다.


In [ ]:
geoai.view_vector_interactive(
    water_gdf,         # 수체 폴리곤 geodataframe
    tiles=raster_url   # 배경지도
)

## 11. QGIS용 결과 저장

탐지 결과를 QGIS에서 열 수 있도록 GeoPackage 형식으로 저장합니다.


In [ ]:
# QGIS용 GeoPackage 저장
water_gdf.to_file("water_prediction.gpkg",layer="water_prediction", driver="GPKG")

In [ ]:
# Colab 파일 다운로드
files.download("water_prediction.gpkg")